# Урок 13 - Памет на агента


## Настройка

Този бележник демонстрира как да се създаде агент за резервации на пътувания с **постоянна памет**, използвайки **Microsoft Agent Framework** (MAF).

Ще научите как различните типове памет на агента — работна, краткосрочна и дългосрочна — влияят на това как агентът запазва и използва информация през разговорите.

**Изисквания:**
- Проект в Azure AI Foundry с разположен чат модел (например `gpt-4o-mini`).
- Вход в Azure CLI — изпълнете `az login` в терминала си.
- `AZURE_AI_PROJECT_ENDPOINT` — крайна точка на вашия проект в Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — името на вашия разположен модел.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Видове памет на агент

AI агентите могат да използват различни видове памет, като всеки от тях служи за различна цел:

### Работна памет
Самата нишка на разговора — съобщенията, обменяни в една сесия. Агентът може да се позовава на по-ранни съобщения в същата нишка, за да поддържа последователност. В MAF това се създава чрез **`agent.create_session()`**, който връща `AgentSession`.

### Краткосрочна памет
Информация, която съхранява за продължителността на една задача или сесия, но не се запазва постоянно. Например, агентът може да събира факти по време на многократен разговор за планиране и да ги използва за изготвяне на финалния план.

### Дългосрочна памет
Предпочитания и факти, които се запазват **през сесиите**. Връщащ се потребител не трябва да повтаря своите диетични ограничения или стил на пътуване. Дългосрочната памет обикновено се поддържа от външен хранилище — база данни, файл или векторен индекс — и се предоставя на агента чрез инструменти.


## Работна памет с сесии

Най-простата форма на памет е сесията на разговора. Когато предавате същата сесия (създадена чрез `agent.create_session()`) на последователни извиквания на `agent.run()`, агентът вижда цялата история на този разговор и може да си припомня по-ранни детайли.

Нека създадем пътнически агент и демонстрираме работната памет.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Агентът правилно си спомни бюджета, защото и двете съобщения споделят една и съща сесия. Това е **работна памет** — съществува само за времето на сесията.

### Какво се случва с нова нишка?

Ако създадем **нова** сесия, агентът няма спомен от предишния разговор:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Модел за дългосрочна памет

За да запомним предпочитанията на потребителя **през различните сесии**, се нуждаем от постоянен склад, който съществува извън нишката на разговора. Агентът достъпва този склад чрез **инструменти** — функции, които може да извиква, за да записва и извлича информация.

По-долу реализираме прост вътрешнопаметен склад за предпочитания (в продукционна среда бихте го поддържали с база данни или векторен индекс) и го предоставяме като инструменти, които агентът може да използва.

### Архитектура
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Сценарий 1 — Потребител за първи път резервира пътуване за годишнина

Сара посещава за първи път. Агентът трябва да съхрани нейните предпочитания чрез инструментите и да ги използва, за да препоръча хотели.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Сценарий 2 — Сара се връща след седмици

Сара започва **чисто нова тема** (симулирайки нова сесия). Работната памет е празна, но хранилището за дългосрочни предпочитания все още съдържа нейната информация. Агентът трябва да я извлече и да я използва за персонализиране на препоръките.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Резюме

В този урок научихте три типа памет на агентите и как да ги имплементирате с Microsoft Agent Framework:

| Тип памет | Механизъм на MAF | Живот |
|---|---|---|
| **Работна** | `agent.create_session()` | Един разговор |
| **Краткотрайна** | Натрупан контекст в рамките на нишка | Една задача / сесия |
| **Дълготрайна** | Външно хранилище, достъпвано чрез `@tool` функции | Между сесии |

### Основни изводи
1. **`agent.create_session()`** предоставя работна памет — агентът вижда цялата история на разговора в рамките на сесия.
2. **Новите сесии губят контекста** — без дълготрайна памет агентът не може да си припомни предишни разговори.
3. **`@tool` функциите** създават мост — те позволяват на агента да записва и извлича информация от постоянно хранилище.
4. **Персонализацията се подобрява с времето** — колкото повече предпочитания се съхраняват, толкова по-добри са препоръките на агента.

### Приложения в реалния свят
- **Обслужване на клиенти**: Запомняне на историята и предпочитанията на клиентите
- **Лични асистенти**: Поддържане на контекст през дни или седмици
- **Здравеопазване**: Следене на информацията и предпочитанията на пациенти
- **Електронна търговия**: Персонализирано пазаруване на база история

### Следващи стъпки
- Замяна на речника в паметта с база данни или векторно хранилище (напр. Azure AI Search)
- Добавяне на изтичане на паметта за чувствителна на време информация
- Изграждане на мултиагентни системи с споделена памет
- Изследване на Cognee тетрадка за памет, базирана на граф знания


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:  
Този документ е преведен с помощта на AI преводаческа услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля, имайте предвид, че автоматичните преводи може да съдържат грешки или неточности. Оригиналният документ на неговия оригинален език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за никакви недоразумения или неправилни тълкувания, възникнали от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
